In [25]:
import yfinance as yf
import datetime as dt
import pandas as pd
import scipy as sp
import mplfinance as mpf
import numpy as np

def ticker_price(ticker):
    data = yf.download(ticker, period='1y', interval='1d').reset_index(level=0, drop=True)
    return data

def ticker_financial(ticker):
    tck = yf.Ticker(ticker)
    return tck.get_financials()

def ticker_options(ticker):
    tck = yf.Ticker(ticker)
    options = tck.option_chain()
    return options.puts, options.calls

def ticker_institutional_holders(ticker):
    tck = yf.Ticker(ticker)
    return tck.institutional_holders

def ticker_support_resistance(ticker, tickerPrice):
    tickerPrice.index = pd.to_datetime(tickerPrice.index)
    mpf.plot(tickerPrice, type='candle', style='charles', title=ticker + 'CandleStick Chart', volume=True)

    # Define the distance between strong peaks (in days).
    strong_peak_distance = 60

    # Define the prominence (how high the peaks are compared to their surroundings).
    strong_peak_prominence = 20

    # Find the strong peaks in the 'high' price data
    strong_peaks, _ = sp.signal.find_peaks(
    tickerPrice['High'],
    distance=strong_peak_distance,
    prominence=strong_peak_prominence
    )

    # Extract the corresponding high values of the strong peaks
    strong_peaks_values = tickerPrice.iloc[strong_peaks]["High"].values.tolist()

    # Include the yearly high as an additional strong peak
    yearly_high = tickerPrice["High"].iloc[-252:].max()
    strong_peaks_values.append(yearly_high)

    add_plot = [mpf.make_addplot(np.full(tickerPrice.shape[0], resistance), color='r', linestyle='--') for resistance in strong_peaks_values]

# Plot the candlestick chart with resistance lines
    mpf.plot(
    tickerPrice, 
    type='candle', 
    style='charles', 
    title=ticker + ' Candlestick Chart with Strong Resistance Lines',
    volume=True, 
    addplot=add_plot
    )
    
    # Define the shorter distance between general peaks (in days)
    # This controls how far apart peaks need to be to be considered separate.
    peak_distance = 5

    # Define the width (vertical distance) where peaks within this range will be grouped together.
    # If the high prices of two peaks are closer than this value, they will be merged into a single resistance level.
    peak_rank_width = 2

    # Define the threshold for how many times the stock has to reject a level
    # Before it becomes a resistance level
    resistance_min_pivot_rank = 3

    # Find general peaks in the stock's 'high' prices based on the defined distance between them.
    # The peaks variable will store the indices of the high points in the 'high' price data.
    peaks, _ = sp.signal.find_peaks(tickerPrice['High'], distance=peak_distance)

    # Initialize a dictionary to track the rank of each peak
    peak_to_rank = {peak: 0 for peak in peaks}

    # Loop through all general peaks to compare their proximity and rank them
    for i, current_peak in enumerate(peaks):
        # Get the current peak's high price
        current_high = tickerPrice.iloc[current_peak]["High"]
        
        # Compare the current peak with previous peaks to calculate rank based on proximity
        for previous_peak in peaks[:i]:
            if abs(current_high - tickerPrice.iloc[previous_peak]["High"]) <= peak_rank_width:
                # Increase rank if the current peak is close to a previous peak
                peak_to_rank[current_peak] += 1

    
    # Initialize the list of resistance levels with the strong peaks already identified.
    resistances = strong_peaks_values

    # Now, go through each general peak and add it to the resistance list if its rank meets the minimum threshold.
    for peak, rank in peak_to_rank.items():
        # If the peak's rank is greater than or equal to the resistance_min_pivot_rank, 
        # it means this peak level has been rejected enough times to be considered a resistance level.
        if rank >= resistance_min_pivot_rank:
            # Append the peak's high price to the resistances list, adding a small offset (1e-3) 
            # to avoid floating-point precision issues during the comparison.
            resistances.append(tickerPrice.iloc[peak]["High"] + 1e-3)

    # Sort the list of resistance levels so that they are in ascending order.
    resistances.sort()

    # Initialize a list to hold bins of resistance levels that are close to each other.
    resistance_bins = []

    # Start the first bin with the first resistance level.
    current_bin = [resistances[0]]

    # Loop through the sorted resistance levels.
    for r in resistances:
        # If the difference between the current resistance level and the last one in the current bin 
        # is smaller than a certain threshold (defined by peak_rank_w_pct), add it to the current bin.
        if r - current_bin[-1] < peak_rank_width:
            current_bin.append(r)
        else:
            # If the current resistance level is far enough from the last one, close the current bin
            # and start a new one.
            resistance_bins.append(current_bin)
            current_bin = [r]

    # Append the last bin.
    resistance_bins.append(current_bin)

    # For each bin, calculate the average of the resistances within that bin.
    # This will produce a clean list of resistance levels where nearby peaks have been merged.
    resistances = [np.mean(bin) for bin in resistance_bins]

    troughs, _ = sp.signal.find_peaks(-tickerPrice['Low'], distance=peak_distance)

In [26]:
#tickers = pd.read_csv('screen.csv')[["symbol"]]
tickers = ['ORCL', 'AMD', 'NVDA', 'PLTR', 'AVGO', 'DIS', 'MSFT']
for ticker in tickers:
    tckPrice = ticker_price(ticker)
    tckPrice.to_csv(ticker+'price.csv')
    tckPuts, tckCalls = ticker_options(ticker)
    tckCalls.to_csv(ticker+'calls.csv')
    tckPuts.to_csv(ticker+'puts.csv')
    tckInstitutionalHolders = ticker_institutional_holders(ticker)
    tckFinancial = ticker_financial(ticker)
    tckFinancial.to_csv(ticker+'financial.csv')
    tckInstitutionalHolders.to_csv(ticker+'institutional.csv')


/var/folders/lv/snnqnwb93sl5s5h0vtgd3d940000gn/T/ipykernel_59218/1937194867.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='1y', interval='1d').reset_index(level=0, drop=True)
[*********************100%***********************]  1 of 1 completed
/var/folders/lv/snnqnwb93sl5s5h0vtgd3d940000gn/T/ipykernel_59218/1937194867.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='1y', interval='1d').reset_index(level=0, drop=True)
[*********************100%***********************]  1 of 1 completed
/var/folders/lv/snnqnwb93sl5s5h0vtgd3d940000gn/T/ipykernel_59218/1937194867.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='1y', interval='1d').reset_index(level=0, drop=True)
[*********************100%***********************]  1 of 1 completed
/var/folders/lv/snnqnwb93sl5s5h0vtgd3d940000gn

In [27]:
tckPuts

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency
0,MSFT250725P00240000,2025-06-12 14:30:05+00:00,240.0,0.19,0.00,0.02,0.000000,0.000000,1.0,1,1.656252,False,REGULAR,USD
1,MSFT250725P00275000,2025-06-12 14:30:05+00:00,275.0,0.17,0.00,0.02,0.000000,0.000000,46.0,46,1.359378,False,REGULAR,USD
2,MSFT250725P00280000,2025-06-16 17:27:06+00:00,280.0,0.16,0.00,0.00,0.000000,0.000000,18.0,18,0.500005,False,REGULAR,USD
3,MSFT250725P00285000,2025-06-23 19:59:45+00:00,285.0,0.05,0.00,0.06,0.000000,0.000000,NaN,2,1.406253,False,REGULAR,USD
4,MSFT250725P00290000,2025-06-30 16:23:46+00:00,290.0,0.14,0.00,0.04,0.000000,0.000000,NaN,54,1.328128,False,REGULAR,USD
5,MSFT250725P00300000,2025-06-26 19:50:46+00:00,300.0,0.02,0.00,0.06,0.000000,0.000000,1.0,4,1.289066,False,REGULAR,USD
6,MSFT250725P00325000,2025-06-16 17:25:26+00:00,325.0,0.17,0.00,0.00,0.000000,0.000000,6.0,6,0.500005,False,REGULAR,USD
7,MSFT250725P00330000,2025-07-14 14:38:00+00:00,330.0,0.01,0.00,0.06,0.000000,0.000000,5.0,14,1.074223,False,REGULAR,USD
8,MSFT250725P00335000,2025-07-07 15:01:45+00:00,335.0,0.03,0.00,0.06,0.000000,0.000000,3.0,43,1.039067,False,REGULAR,USD
9,MSFT250725P00340000,2025-07-10 14:00:27+00:00,340.0,0.02,0.00,0.06,0.000000,0.000000,NaN,10,1.007817,False,REGULAR,USD
